<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 28 · GenAI Workflows for Fixed Income

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch28_genai_workflows_fixed_income.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

This chapter establishes the verification-first pattern for using generative AI in fixed-income work. You will learn to separate language generation from numerical control: GenAI may draft, classify, and extract, while Python owns numbers, schemas, and reconciliation. You will define an explicit output schema before any draft is produced, route low-confidence or flagged fields into a review queue, and reconcile every drafted value against a stored source table before distribution. You will also recognize the characteristic failure modes, such as changed numbers, reversed signs, and unsupported causal claims, and respond by fixing the evidence chain rather than asking for more fluent prose.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script works entirely on frozen, synthetic records; no model is called and no network is used. It builds a review queue from the stored extraction table, sending every field with confidence below 0.85 or an explicit review flag to a human, and then summarizes the stored draft-to-source validation checks. Two fields queue for review and one drafted value fails reconciliation, which is the workflow's central lesson.

In [ ]:
SCRIPT = ROOT / "code" / "ch28_genai_workflows_fixed_income.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Identify the two queued fields and why: spread_change (confidence 0.81, review flag set) and weekly_outflow (confidence 0.79, review flag set).
Note that high confidence alone does not clear a field: all five extractions still pass through the validation table, where four checks pass and one fails.
Locate the failed check: the drafted spread change of 16.0 bp does not match the extracted 18.0 bp.
Confirm that the correct response is to correct or remove the unsupported statement, not to regenerate more fluent prose.
Remember that every record is frozen synthetic input; the script makes no live model call and reads no real document.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch28_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Structured extraction with schema validation

This lab isolates one mechanism from the chapter's control loop: parsing named fields out of a mock note and validating each against a schema of expected units and plausible ranges. The note text is synthetic; a field that fails the presence check or the range check is routed to review instead of into the report.

In [ ]:
import re

note = "DV01 4.8 USD m/bp. Spread change +78.0 bp."  # mock note
schema = {  # field: regex, unit, low, high
    "dv01": (r"DV01 ([0-9.]+)", "USD m/bp", 0.0, 10.0),
    "spread_change": (r"Spread change \+([0-9.]+)", "bp", -50.0, 50.0),
    "cash_buffer": (r"cash buffer ([0-9.]+)", "USD m", 0.0, 100.0),
}
for field, (pat, unit, low, high) in schema.items():
    m = re.search(pat, note)  # parse the field
    if m is None:  # presence check
        print(f"{field}: missing -> review")
        continue
    v = float(m.group(1))
    status = "pass" if low <= v <= high else "review"  # range check
    print(f"{field}: {v} {unit} -> {status}")

### What to observe

Expected output: dv01 4.8 USD m/bp passes, spread_change 78.0 bp fails the plus/minus 50 bp range and is routed to review, and cash_buffer is absent from the note entirely and is also routed to review. Two of three fields never reach the report unchanged; extraction without validation is not evidence.

## Exercises

1. **Queue audit.** Rebuild the review queue at a confidence threshold of 0.75 instead of 0.85 and record which fields change status. *Hint:* Only the confidence comparison changes; the needs_review flag is independent of the threshold.
2. **Reconciliation.** Join ch28_structured_extraction.csv to ch28_report_validation.csv and print each field's source value, draft value, and status side by side. *Hint:* spread_change is the field where 18.0 bp became 16.0 bp in the draft.
3. **Schema design.** Add a ten_year_yield_change field with a plausible bp range to the mini-lab schema and test a note in which the field is absent. *Hint:* The presence check routes missing fields to review before any range check can run.
4. **Failure modes.** For each row of the validation table, name the failure mode the chapter warns about and the control that catches it. *Hint:* A model can keep the unit but change the number; only exact-value reconciliation catches that.
5. **Scope check.** Explain why a report may say the spread widened 18 bp but may not say the issuer is deteriorating unless separate credit evidence supports it. *Hint:* Scope discipline is part of the chapter's control design, not a style preference.

For more detail, see Chapter 28 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">